# 02. Feature Engineering

EDA 결과를 바탕으로 모델 입력 피처를 생성하고 검증합니다.

## 00. 경로 설정
프로젝트 폴더 기반으로 실행 경로 지정

1. 프로젝트 루트에서 시작하면 현재 폴더 사용
2. notebooks 폴더에서 실행하면 한 단계 위 사용

In [1]:
from pathlib import Path
import sys

import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("프로젝트 폴더를 찾을 수 없습니다.")

# src의 Python 파일을 import할 수 있도록 경로 등록
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess import (
    BATCH_1_PATH,
    load_mat,
    to_list_of_dicts,
    extract_summary,
    extract_cycle_qdlin,
)
from src.features import build_feature_table

print("프로젝트 경로:", PROJECT_ROOT)
print("커널 실행 파일:", sys.executable)

프로젝트 경로: /Users/sonsugyung/skala/11th_week/data-mini-prj
커널 실행 파일: /Users/sonsugyung/skala/11th_week/data-mini-prj/.venv/bin/python


## 01. 데이터 로드
필요한 데이터(Batch 1)을 가져와 저장  
이후 필드별 데이터를 셀별 딕셔너리 목록으로 변환

In [2]:
print("Batch 1 로딩 중...")
mat = load_mat(BATCH_1_PATH)

batch = to_list_of_dicts(mat["batch"])

print("셀 개수:", len(batch))

ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:root:ERROR: MATLAB type not supported: string, (uint32)
ERROR:ro

Batch 1 로딩 중...
셀 개수: 46


## 02. 전처리 결과 생성

사이클별 요약 데이터를 표로 정리  
ΔQ 계산에 사용할 두 사이클의 용량 곡선을 추출  
preprocess.py의 함수를 호출하여 추출

In [3]:
# 사이클별 요약 데이터를 표로 정리
summary_df = extract_summary(batch)

# 두 사이클의 용량 곡선을 추출
qd10_df = extract_cycle_qdlin(batch, 10)
qd100_df = extract_cycle_qdlin(batch, 100)

print("Summary 크기:", summary_df.shape)
print("Cycle 10 추출 셀 수:", len(qd10_df))
print("Cycle 100 추출 셀 수:", len(qd100_df))

display(summary_df.head())

Summary 크기: (38811, 11)
Cycle 10 추출 셀 수: 46
Cycle 100 추출 셀 수: 46


,cell_id,cycle_life,charging_policy,cycle,QD,QC,IR,Tmax,Tavg,Tmin,chargetime
0,0,1190.0,3.6C(80%)-3.6C,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,1190.0,3.6C(80%)-3.6C,2.0,1.070689,1.071042,0.016742,35.652016,31.875011,29.566130,13.341250
2,0,1190.0,3.6C(80%)-3.6C,3.0,1.071900,1.071674,0.016724,35.692978,31.931490,29.604385,13.425777
3,0,1190.0,3.6C(80%)-3.6C,4.0,1.072510,1.072304,0.016681,35.680588,31.932603,29.744202,13.425167
4,0,1190.0,3.6C(80%)-3.6C,5.0,1.073174,1.072970,0.016662,35.728691,31.959322,29.644709,13.341442


## 03. 피처 생성
log_delta_q_variance: Cycle 100−10의 Qdlin 차이에 대한 로그 분산  
mean_chargetime: 초기 100사이클 평균 충전 시간  
EDA와 동일한 계산 기준 적용  
셀당 한 행으로 두 피처와 cycle_life 연결

In [4]:
# features.py의 함수를 호출해 두 피처와 Target을 연결
feature_table = build_feature_table(
    summary_df,
    qd10_df,
    qd100_df,
)

print("피처 표 크기:", feature_table.shape)
display(feature_table.head())

피처 표 크기: (46, 4)


,cell_id,log_delta_q_variance,mean_chargetime,cycle_life
0,0,-5.014427,13.250388,1190.0
1,1,-5.013525,13.240949,1179.0
2,2,-4.736565,13.245097,1177.0
3,3,-4.442179,11.932744,1226.0
4,4,-4.647309,11.932781,1227.0


## 04. 피처 검증
셀 수와 열 구성 확인  
cell_id 결측, 중복 확인  
피처와 Target의 결측값, 무한대 확인  
Cycle Life의 양수 여부 확인  

In [5]:
expected_columns = [
    "cell_id",
    "log_delta_q_variance",
    "mean_chargetime",
    "cycle_life",
]

# 예상한 행 수와 열 구성을 검사
assert feature_table.shape == (46, 4), "피처 표 크기가 다릅니다."
assert feature_table.columns.tolist() == expected_columns, "열 구성이 다릅니다."

# 셀당 한 행인지 검사
assert feature_table["cell_id"].notna().all(), "cell_id에 결측값이 있습니다."
assert feature_table["cell_id"].is_unique, "cell_id가 중복됩니다."

# 피처와 Target이 모두 유한한 숫자인지 검사
numeric_columns = expected_columns[1:]
assert np.isfinite(
    feature_table[numeric_columns].to_numpy(dtype=float)
).all(), "결측값 또는 무한대가 있습니다."

assert feature_table["cycle_life"].gt(0).all(), "Target에 0 이하 값이 있습니다."

print("피처 표 기본 검사 통과")
display(feature_table[numeric_columns].describe().round(4))

피처 표 기본 검사 통과


,log_delta_q_variance,mean_chargetime,cycle_life
count,46.0000,46.0000,46.0000
mean,-3.9230,11.1683,844.7174
std,0.3783,1.3949,184.6292
min,-5.0144,8.8974,534.0000
25%,-4.0530,10.0599,703.2500
50%,-3.8507,11.0336,858.5000
75%,-3.6678,11.8423,914.2500
max,-3.3499,14.9250,1227.0000


## 05. EDA 계산식 재현 확인
EDA의 계산식을 직접 적용한 값과 함수 반환값 비교  
표본 분산(ddof=1)과 분산 0 처리 기준 유지  
초기 100사이클 평균에 충전 시간 0 포함

In [6]:
# EDA 평균 충전 시간 계산식 직접 적용
early = summary_df[summary_df["cycle"] <= 100]
reference_charge = early.groupby("cell_id")["chargetime"].mean()

# 같은 셀의 Cycle 10·100 곡선 연결
paired = qd10_df[["cell_id", "Qdlin"]].merge(
    qd100_df[["cell_id", "Qdlin"]],
    on="cell_id",
    suffixes=("_10", "_100"),
    validate="one_to_one",
).set_index("cell_id")

# EDA ΔQ 로그 분산 계산식 직접 적용
reference_delta = paired.apply(
    lambda row: np.log10(
        max(
            float(np.var(row["Qdlin_100"] - row["Qdlin_10"], ddof=1)),
            np.finfo(float).tiny,
        )
    ),
    axis=1,
)

# 셀 식별자를 기준으로 순서를 맞춰 비교
actual = feature_table.set_index("cell_id")

np.testing.assert_allclose(
    actual["log_delta_q_variance"],
    reference_delta.reindex(actual.index),
)

np.testing.assert_allclose(
    actual["mean_chargetime"],
    reference_charge.reindex(actual.index),
)

print("두 피처 모두 EDA 계산식과 일치합니다.")

두 피처 모두 EDA 계산식과 일치합니다.


## 06. 결과 요약
Batch 1 : 46개 셀  
최종 표 : 46행 × 4열  
입력 피처 : log_delta_q_variance, mean_chargetime  
Target : cycle_life  
식별자 중복·결측값·무한대 : 없음  
EDA 계산식 재현 검사 : 통과  
Cycle 1의 충전 시간 0 포함 : 기존 EDA 기준 유지